In [1]:
import pandas as pd
import numpy as np

dim_customer = pd.read_parquet("../data/processed/dim_customer.parquet")
fact_loans = pd.read_parquet("../data/processed/fact_loans.parquet")
fact_transactions = pd.read_parquet("../data/processed/fact_transactions.parquet")


# DIM_CUSTOMER – CUSTOMER INTELLIGENCE

In [2]:
dim_customer['customer_risk_band'] = pd.cut(
    dim_customer['avg_debt_ratio'],
    bins=[-1, 50, 150, np.inf],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

dim_customer["customer_risk_band"].head()

0    High Risk
1    High Risk
2    High Risk
3    High Risk
4    High Risk
Name: customer_risk_band, dtype: category
Categories (3, object): ['Low Risk' < 'Medium Risk' < 'High Risk']

In [3]:
dim_customer["late_payment_flag"] = np.where(
    dim_customer["total_late_events"] > 0, 1, 0
)


dim_customer["late_payment_flag"].head()

0    1
1    1
2    0
3    1
4    1
Name: late_payment_flag, dtype: int64

In [4]:
dim_customer["credit_health_score"]=(
    100
    - dim_customer["avg_debt_ratio"].clip(0,100)
    -dim_customer["max_90_days_late"]*5 #5marks penalty
)
dim_customer["credit_health_score"].head()

0     0.0
1   -10.0
2    -5.0
3     0.0
4     0.0
Name: credit_health_score, dtype: float64

In [5]:
dim_customer["customer_value_segment"] = pd.cut(
    dim_customer["balance"],
    bins=[-1, 0, 100000, np.inf],
    labels=["Zero Balance", "Low Value", "High Value"]
)
dim_customer["customer_value_segment"].head()

0    Zero Balance
1       Low Value
2      High Value
3    Zero Balance
4      High Value
Name: customer_value_segment, dtype: category
Categories (3, object): ['Zero Balance' < 'Low Value' < 'High Value']

# FACT_LOANS – LOAN RISK METRICS

In [6]:
fact_loans["loan_to_income_ratio"] = (
    fact_loans["AMT_CREDIT"] / fact_loans["AMT_INCOME_TOTAL"]
)

fact_loans["loan_to_income_ratio"].head()

0    2.007889
1    4.790750
2    2.000000
3    2.316167
4    4.222222
Name: loan_to_income_ratio, dtype: float64

In [7]:
fact_loans["emi_to_income_ratio"] = (
    fact_loans["AMT_ANNUITY"] / fact_loans["AMT_INCOME_TOTAL"]
)

fact_loans["emi_to_income_ratio"].head()

0    0.121978
1    0.132217
2    0.100000
3    0.219900
4    0.179963
Name: emi_to_income_ratio, dtype: float64

In [8]:
fact_loans["loan_risk_flag"] = np.where(
    (fact_loans["loan_to_income_ratio"] > 4) | 
    (fact_loans["TARGET"] == 1),
    1, 0
)

fact_loans["loan_risk_flag"].head()

0    1
1    1
2    0
3    0
4    1
Name: loan_risk_flag, dtype: int64

# FACT_TRANSACTIONS – BEHAVIOR FEATURES

In [9]:
fact_transactions["spend_to_balance_ratio"] = (
    fact_transactions["total_spend"] /
    fact_transactions["last_account_balance"].replace(0, np.nan)
)

fact_transactions["spend_to_balance_ratio"].head()

0    14.000289
1     1.670976
2     1.546910
3     8.442145
4     8.718130
Name: spend_to_balance_ratio, dtype: float64

In [10]:
fact_transactions["high_activity_flag"] = np.where(
    fact_transactions["txn_count"] >
    fact_transactions["txn_count"].quantile(0.75),
    1, 0
)

fact_transactions["high_activity_flag"].head()

0    0
1    0
2    0
3    1
4    0
Name: high_activity_flag, dtype: int64

In [11]:
# Export final SQL-ready datasets

dim_customer.to_csv(
    "../data/sql_final_set/dim_customer.csv",
    index=False
)

fact_loans.to_csv(
    "../data/sql_final_set/fact_loans.csv",
    index=False
)

fact_transactions.to_csv(
    "../data/sql_final_set/fact_transactions.csv",
    index=False
)

print("SQL final CSV files exported successfully ✅")


SQL final CSV files exported successfully ✅


In [14]:
fact_transactions.columns

Index(['customer_id', 'txn_count', 'total_spend', 'avg_txn_amount',
       'max_txn_amount', 'last_account_balance', 'spend_to_balance_ratio',
       'high_activity_flag'],
      dtype='object')

In [15]:
fact_loans.columns

Index(['customer_id', 'SK_ID_CURR', 'AMT_CREDIT', 'AMT_ANNUITY',
       'AMT_GOODS_PRICE', 'AMT_INCOME_TOTAL', 'TARGET', 'loan_to_income_ratio',
       'emi_to_income_ratio', 'loan_risk_flag'],
      dtype='object')